# 11 — Distress-language estimator (Tier 3, fourth signal)

**Read this before running.** Notebook 00 kept only 12 columns from `objects.csv`
and dropped the text fields. The Crunchbase dump *does* carry company text:
`short_description`, `description`, `overview`, `tag_list`. This notebook goes
back to the raw file for those columns and builds the fourth Tier-3 estimator
from them.

**What this is, honestly:**
- It IS real text tied to real outcomes, weak-labelled by whether the company
  closed — the distant-supervision approach of Mintz et al. / Ratner et al.
- It is NOT the founder/press corpus the IDF describes. Company descriptions
  are static marketing copy with **no publication date**, so the
  announcement-exclusion window (IDF paragraph [0018]) cannot be applied —
  there is no "written N months before shutdown" to exclude.
- Therefore this estimator answers *"does the kind of company a description
  describes correlate with failure?"* and NOT *"is this founder's language
  showing distress right now?"* Report it that way. The second question still
  needs a dated founder/press corpus.

Reads: `data/raw/objects.csv`, `data/processed/outcomes.csv`
Writes: `data/processed/distress_scores.csv`, `reports/experiment4_distress_language.csv`

In [1]:
import pandas as pd
import numpy as np
import re, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report

RAW="../data/raw"; PROCESSED="../data/processed"; REPORTS="../reports"
os.makedirs(REPORTS, exist_ok=True)
RANDOM_STATE=42

TEXT_COLS = ["short_description","description","overview","tag_list"]

header = pd.read_csv(f"{RAW}/objects.csv", encoding="ISO-8859-1", nrows=0)
available = [c for c in TEXT_COLS if c in header.columns]
print("text columns present in objects.csv:", available)
if not available:
    raise SystemExit("No text columns found. Check the objects.csv schema before continuing.")

text columns present in objects.csv: ['short_description', 'description', 'overview', 'tag_list']


In [2]:
usecols = ["id","entity_type"] + available
obj = pd.read_csv(f"{RAW}/objects.csv", encoding="ISO-8859-1", usecols=usecols, low_memory=False)
obj = obj[obj["entity_type"]=="Company"].drop(columns=["entity_type"])

obj["text"] = obj[available].fillna("").agg(" ".join, axis=1).str.strip()
obj = obj[obj["text"].str.len() >= 30]          # drop near-empty descriptions
print(f"[text] {len(obj):,} companies carry usable text (>=30 chars)")
print(f"[text] median length: {obj['text'].str.len().median():.0f} chars")
obj[["id","text"]].head(3)

[text] 129,625 companies carry usable text (>=30 chars)
[text] median length: 511 chars


,id,text
0,c:1,Technology Platform Company Wetpaint is a tech...
1,c:10,Flektor is a rich-media mash-up platform that ...
2,c:100,There.com is an online virtual world where any...


## Weak labelling from outcomes, and leakage control

The label is the company's eventual fate — distant supervision. Before
training, any token that *states* the outcome is stripped, otherwise the
model trivially learns "the word 'defunct' means closed" and reports a
meaningless score.

In [3]:
outcomes = pd.read_csv(f"{PROCESSED}/outcomes.csv")
df = obj.merge(outcomes[["id","event","status","category_code"]], on="id", how="inner")
print(f"[join] {len(df):,} companies have both text and a resolved outcome")
print(df["event"].value_counts(normalize=True).rename("proportion"))

# non-capturing group: a capturing group makes pandas .str.contains warn
LEAK = r"\b(?:closed|shut\s*down|shutdown|defunct|acquired|acquisition|ceased|bankrupt|" \
       r"bankruptcy|liquidat\w*|wound\s*down|dissolved|out\s*of\s*business|ipo|" \
       r"went\s*public|delisted|dead\s*pool|deadpool|no\s*longer\s*(?:in\s*)?(?:operat\w*|active)|" \
       r"formerly\s*known|now\s*part\s*of|merged\s*(?:with|into)|rebranded|" \
       r"was\s*bought|bought\s*by|taken\s*over)\b"
def scrub(t):
    t = t.lower()
    t = re.sub(LEAK, " ", t)
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

before = df["text"].str.lower().str.contains(LEAK, regex=True).mean()
df["clean"] = df["text"].map(scrub)
print(f"[leakage] {100*before:.1f}% of descriptions contained an outcome-stating token; all removed")

[join] 96,360 companies have both text and a resolved outcome
event
0    0.902854
1    0.097146
Name: proportion, dtype: float64
[leakage] 4.0% of descriptions contained an outcome-stating token; all removed


## Train the adapted estimator (TF-IDF + logistic regression)

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(
    df["clean"], df["event"], test_size=0.25,
    random_state=RANDOM_STATE, stratify=df["event"])

vec = TfidfVectorizer(max_features=40000, ngram_range=(1,2),
                      min_df=3, sublinear_tf=True, stop_words="english")
Xtr = vec.fit_transform(X_tr); Xte = vec.transform(X_te)
print(f"[vectorise] vocabulary {len(vec.vocabulary_):,} terms")

clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
clf.fit(Xtr, y_tr)

p_te = clf.predict_proba(Xte)[:,1]
yhat = (p_te >= 0.5).astype(int)
adapted = {
    "auc": roc_auc_score(y_te, p_te),
    "macro_f1": f1_score(y_te, yhat, average="macro"),
    "precision": precision_score(y_te, yhat, zero_division=0),
    "recall": recall_score(y_te, yhat, zero_division=0),
}
print(f"[adapted] AUC {adapted['auc']:.3f} | macro-F1 {adapted['macro_f1']:.3f} "
      f"| precision {adapted['precision']:.3f} | recall {adapted['recall']:.3f}")

[vectorise] vocabulary 40,000 terms
[adapted] AUC 0.726 | macro-F1 0.580 | precision 0.211 | recall 0.506


## Experiment 4 — compare against a fixed financial lexicon

The IDF specifies comparison against a fixed financial sentiment lexicon.
A compact Loughran–McDonald-style negative word list stands in for the full
dictionary; the point of the comparison is that a *fixed, general* finance
lexicon underperforms a corpus-*adapted* model, which is exactly Loughran &
McDonald's own argument applied one level further.

In [5]:
LM_NEGATIVE = set("""
adverse adversely against bad challenge challenged challenging concern concerns
control costly crisis critical damage decline declined declining deficit
difficult difficulties difficulty discontinued dispute doubt downturn fail
failed failing failure fails fear force forced hurt impair impaired impairment
inability incur incurred lack late litigation lose loses losing loss losses
negative obsolete penalty plaintiff poor problem problems recall restructuring
risk risks risky serious sever severe slow slowdown struggle termination
threat uncertain uncertainty unable unfavorable unpaid volatile weak weakness
""".split())

def lexicon_score(t):
    toks = t.split()
    return sum(w in LM_NEGATIVE for w in toks) / max(len(toks), 1)

lex_te = X_te.map(lexicon_score).values
lex_auc = roc_auc_score(y_te, lex_te)
print(f"[lexicon baseline] AUC {lex_auc:.3f}")
print(f"[adapted model]    AUC {adapted['auc']:.3f}")
print()
if adapted["auc"] > lex_auc:
    print(f"Adapted model beats the fixed lexicon by {adapted['auc']-lex_auc:+.3f} AUC.")
else:
    print("Adapted model does NOT beat the fixed lexicon -- report that plainly.")

[lexicon baseline] AUC 0.510
[adapted model]    AUC 0.726

Adapted model beats the fixed lexicon by +0.216 AUC.


In [6]:
# what the model actually keys on -- sanity check that it is not junk
names = np.array(vec.get_feature_names_out()); coefs = clf.coef_[0]
top_risk  = names[np.argsort(coefs)[-18:]][::-1]
top_safe  = names[np.argsort(coefs)[:18]]
print("terms most associated with CLOSURE:\n ", ", ".join(top_risk))
print("\nterms most associated with SURVIVAL:\n ", ", ".join(top_safe))
print()
print("Inspect these. If they look like outcome words that slipped past the scrub,")
print("extend LEAK above and re-run. If they look like sector/business-model words,")
print("the model is reading company type -- which is the honest interpretation.")

terms most associated with CLOSURE:
  founded based, company, headquartered, iphone blackberry, enables, founded headquartered, france, entered, beijing, combinator, web ireland, handset, left, china, centres, mobile phone, university, gmbh

terms most associated with SURVIVAL:
  big data, ios, seo, app, latest, cloud based, beautiful, custom, cloud, advertise, operates subsidiary, consultancy, tablet, web development, clients, report, instagram, fundraising

Inspect these. If they look like outcome words that slipped past the scrub,
extend LEAK above and re-run. If they look like sector/business-model words,
the model is reading company type -- which is the honest interpretation.


## Score every company and emit the fusion signal

In [7]:
# IMPORTANT: scores must be OUT-OF-FOLD. Scoring every company with a model that
# was trained on that company is in-sample, and feeding such a score into the
# fusion of notebook 12 -- evaluated against the same labels -- would leak the
# label and make this signal look far stronger than it is.
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold

pipe = make_pipeline(
    TfidfVectorizer(max_features=40000, ngram_range=(1,2), min_df=3,
                    sublinear_tf=True, stop_words="english"),
    LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0))

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
df["distress_score"] = cross_val_predict(pipe, df["clean"], df["event"],
                                          cv=cv5, method="predict_proba")[:,1]
oof_auc = roc_auc_score(df["event"], df["distress_score"])
print(f"[out-of-fold] AUC across all companies: {oof_auc:.3f}  <- use THIS figure, not the held-out one above")
out = df[["id","distress_score"]].rename(columns={"id":"object_id"})
out.to_csv(f"{PROCESSED}/distress_scores.csv", index=False)
print(f"[done] wrote {PROCESSED}/distress_scores.csv for {len(out):,} companies")
print(f"[done] mean distress score: {out['distress_score'].mean():.3f}")

pd.DataFrame([
    {"model":"Adapted TF-IDF + logistic (this work)","auc":round(adapted["auc"],3),
     "macro_f1":round(adapted["macro_f1"],3),"note":"weak supervision from outcomes"},
    {"model":"Adapted model, out-of-fold","auc":round(oof_auc,3),
     "macro_f1":np.nan,"note":"5-fold OOF -- the score exported for fusion"},
    {"model":"Fixed financial lexicon","auc":round(lex_auc,3),
     "macro_f1":np.nan,"note":"Loughran-McDonald-style negative word rate"},
]).to_csv(f"{REPORTS}/experiment4_distress_language.csv", index=False)
print(f"[done] wrote {REPORTS}/experiment4_distress_language.csv")

[out-of-fold] AUC across all companies: 0.722  <- use THIS figure, not the held-out one above
[done] wrote ../data/processed/distress_scores.csv for 96,360 companies
[done] mean distress score: 0.345
[done] wrote ../reports/experiment4_distress_language.csv


**For the report.** State the limitation in the same sentence as the result:
the estimator is trained on undated company self-descriptions, so it captures
company-type signal rather than temporal distress, and the exclusion-window
mechanism of IDF [0018] remains untested pending a dated corpus. Run
notebook 12 next to fold this signal into the fusion and see whether the
four-signal RHI finally beats the survival signal alone.